# Vignette 4: Augment a classifier with the public synthetic dataset

The 2024 release `tbuckley/synthetic-derm-1M` holds about one million synthetic images in
WebDataset shards named by label and generation method. This notebook pulls the text-to-image
shards for the six sample labels, keeps only images generated from the sample's own training
photographs (so validation images never leak through their synthetic copies), trains a timm
classifier with and without them, and reports accuracy with the paper's Wald confidence
intervals.

Set `SYNTHETIC_DIR` to a folder written by `generate_synthetic_dataset` (for example the output
of vignette 2 or 3) to use your own images instead of the 2024 shards.

In [ ]:
SMALL = True
SYNTHETIC_DIR = None          # e.g. "/workspace/synderm/generations/qwen-lora-t2i"
N_REAL_PER_CLASS = 16
N_SYNTHETIC_PER_REAL = 10
MAX_EPOCHS = 8 if SMALL else 50
BACKBONE = "tf_efficientnetv2_m.in1k"   # the paper's backbone; try "convnext_tiny.fb_in22k_ft_in1k"
SEED = 0

In [ ]:
import os, json, time
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image

os.environ.setdefault("HF_HOME", "/workspace/.cache/huggingface")
WORK = Path(os.environ.get("SYNDERM_WORK_DIR", "/workspace/synderm"))
WORK.mkdir(parents=True, exist_ok=True)
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

def show_grid(rows, row_titles=None, col_titles=None, size=2.6, suptitle=None):
    """rows: list of lists of PIL images (one list per row)."""
    n_rows, n_cols = len(rows), max(len(r) for r in rows)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(size * n_cols, size * n_rows), squeeze=False)
    for r, row in enumerate(rows):
        for c in range(n_cols):
            ax = axes[r][c]; ax.axis("off")
            if c < len(row) and row[c] is not None:
                ax.imshow(row[c])
            if r == 0 and col_titles and c < len(col_titles):
                ax.set_title(col_titles[c], fontsize=10)
        if row_titles:
            axes[r][0].text(-0.08, 0.5, row_titles[r], transform=axes[r][0].transAxes, rotation=90,
                            va="center", ha="right", fontsize=10)
    if suptitle:
        fig.suptitle(suptitle)
    plt.tight_layout(); plt.show()

In [ ]:
from synderm.data import load_sample, load_2024_shards, SyntheticDataset
from synderm.data.atlas import HELD_OUT_CSV
real_train, real_val = load_sample(split="train"), load_sample(split="val")
classes = real_train.labels
# Synthetic images derived from validation or held-out real images are excluded (no leakage).
banned = set(real_val.df["md5hash"]) | set(pd.read_csv(HELD_OUT_CSV)["md5hash"])
POLICY = "ref-in-train" if SYNTHETIC_DIR else "ref-not-in-val"
synthetic = SyntheticDataset(SYNTHETIC_DIR, labels=classes) if SYNTHETIC_DIR else load_2024_shards(
    classes, method="finetune-text-to-image", submethod="text-to-image", max_per_label=N_SYNTHETIC_PER_REAL * 64,
    exclude_ref_ids=banned)
print(len(real_train), "real train |", len(real_val), "real held-out |", len(synthetic), "synthetic")
show_grid([[synthetic[i]["image"] for i in range(0, len(synthetic), max(1, len(synthetic) // 6))][:6]], suptitle="synthetic examples")

## Split and mix

`synthetic_train_val_split` takes `per_class_test_size` real images per class for validation
(model selection), caps real training images at `N_REAL_PER_CLASS` and attaches only synthetic
images whose `ref_id` is a real training image, up to `N_SYNTHETIC_PER_REAL` each. The
held-out set from `load_sample(split="val")` is used once, for the final numbers.

In [ ]:
from synderm.utils import synthetic_train_val_split
from synderm.classify import ClassificationDataset, build_transforms, train_classifier
def run(with_synthetic, seed=SEED):
    train, val = synthetic_train_val_split(real_train, synthetic if with_synthetic else None, per_class_test_size=8,
                                           n_real_per_class=N_REAL_PER_CLASS, n_synthetic_per_real=N_SYNTHETIC_PER_REAL,
                                           random_state=seed, synthetic_policy=POLICY, exclude_ref_ids=banned)
    res = train_classifier(ClassificationDataset(train, classes, build_transforms(augment=True)),
                           ClassificationDataset(val, classes), classes,
                           test_dataset=ClassificationDataset(real_val, classes), backbone=BACKBONE,
                           max_epochs=MAX_EPOCHS, seed=seed, verbose=False)
    m = res.test_metrics
    print(f"synthetic={with_synthetic!s:5s} n_train={len(train):5d} best epoch {res.best_epoch}  test acc {m['accuracy']:.3f} [{m['ci_low']:.3f}, {m['ci_high']:.3f}]")
    return res
baseline = run(False)
augmented = run(True)

In [ ]:
fig, ax = plt.subplots(figsize=(4, 3.5))
for x, (name, r) in enumerate({"real only": baseline, "real + synthetic": augmented}.items()):
    m = r.test_metrics
    ax.bar(x, m["accuracy"], yerr=[[m["accuracy"] - m["ci_low"]], [m["ci_high"] - m["accuracy"]]], capsize=4, label=name)
ax.set_xticks([0, 1]); ax.set_xticklabels(["real only", "real + synthetic"]); ax.set_ylabel("held-out accuracy")
ax.set_title(f"{N_REAL_PER_CLASS} real per class, {len(classes)} classes"); plt.tight_layout(); plt.show()
pd.DataFrame({k: {c: v["accuracy"] for c, v in r.test_metrics["per_class"].items()} for k, r in {"real only": baseline, "real + synthetic": augmented}.items()}).round(3)

With one seed and a small sample the difference can go either way; the paper averaged five
runs. Vignette 5 sweeps the number of real images and repeats seeds, which is where the
low-data effect shows.